# DLNet baselines: train, compare and reuse

This notebook belongs to the DLNet output folder it sits in (3 baselines for a **classification** task,
metric **accuracy**). It runs on a laptop (Jupyter) or on **Google Colab** (GPU: *Runtime > Change
runtime type > GPU*).

Every baseline is a self-contained package `top<k>_<family>/`: `model.py` is the network as plain
PyTorch (no DLNet import), `config.yaml` its hyper-parameters, `architecture.svg` its block diagram.

1. **Setup**: get the folder onto Colab, install what is missing, choose the device.
2. **Data**: the same file, split and preprocessing as the search.
3. **One section per baseline**: block diagram, layer graph (torchview), full training, curves.
4. **Comparison** of the baselines.
5. **Reuse**: the saved weights loaded back and used on the data.

The search ranked the baselines on a short budget; full training can reorder neighbours.

In [ ]:
# 1. Setup ---------------------------------------------------------------------------------
# Local Jupyter: open this notebook inside the DLNet output folder; nothing to change here.
# Google Colab: the output folder (and the data) must be on the Colab machine first. Choose:
#   SOURCE = "git"  -> clone GIT_URL and go to GIT_SUBDIR (a private repo needs a token in the URL)
#   SOURCE = "zip"  -> upload a .zip of the output folder (with the data inside or next to it)
import os, subprocess, sys, zipfile
from pathlib import Path

SOURCE = 'git'          # "here" | "git" | "zip"
GIT_URL = 'https://github.com/Duy-Huan/DLNet.git'
GIT_SUBDIR = 'examples/hsi/botswana/output'
IN_COLAB = "google.colab" in sys.modules

if SOURCE == "git" and not Path("leaderboard.json").exists():
    repo = Path("/content/repo" if IN_COLAB else "repo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(repo)], check=True)
    os.chdir(repo / GIT_SUBDIR)
elif SOURCE == "zip" and not Path("leaderboard.json").exists():
    from google.colab import files  # Colab only
    for name in files.upload():
        zipfile.ZipFile(name).extractall(".")
    os.chdir(next(p.parent for p in Path(".").rglob("leaderboard.json")))
OUT = Path.cwd()
assert (OUT / "leaderboard.json").exists(), f"{OUT} is not a DLNet output folder (no leaderboard.json)"

# what the packages need; torch, numpy, pandas, scipy and matplotlib are already on Colab
needed = {'torchview': 'torchview', 'graphviz': 'graphviz', 'matplotlib': 'matplotlib', 'scipy': 'scipy'}
missing = []
for mod, pip_name in needed.items():
    try:
        __import__(mod)
    except ImportError:
        missing.append(pip_name)
if missing and os.environ.get("DLNET_NO_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

import numpy as np
import torch

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("output folder:", OUT)
print("torch", torch.__version__, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
# Helpers: every package has its own model.py / core.py / inference.py ..., so one package is
# imported at a time. `use(pkg)` returns its modules.
import importlib, json, time, types

SHARED = ("model", "core", "inference", "array_data", "hsi_data")
_ACTIVE = []

def use(pkg):
    pkg = (OUT / pkg).resolve()
    for name in SHARED:
        sys.modules.pop(name, None)
    while _ACTIVE:
        old = _ACTIVE.pop()
        if old in sys.path:
            sys.path.remove(old)
    sys.path.insert(0, str(pkg))
    _ACTIVE.append(str(pkg))
    mods = types.SimpleNamespace(dir=pkg, cfg=json.loads((pkg / "config.yaml").read_text()))
    for name in SHARED:
        if (pkg / f"{name}.py").exists():
            setattr(mods, name, importlib.import_module(name))
    return mods

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
try:
    from IPython.display import SVG, Markdown, display
except ImportError:  # plain Python (no IPython): say where the picture is
    def SVG(filename=None, data=None):
        return f"[picture: {filename}]"
    Markdown = str
    def display(x):
        print(x)

In [ ]:
# 0. What the search found
board = json.loads(Path("leaderboard.json").read_text())
BASELINES = [r["name"] for r in board["top_k"]]
print("task:", board["task"], "| metric:", board["metric"])
for r in board["top_k"]:
    print(f"  {r['name']:<45} params {r['params']:>10,}   val {r['val_mean']:.4f} +- {r['val_std']:.4f}"
          f"   (search: {r['epochs']} epochs)")
print("verdict:", board.get("verdict"))

## 2. Data

Edit the path(s) if the data is somewhere else on this machine.

In [ ]:
# 2. Data: the same cube, split, PCA and patches as the search (the package's hsi_data.py)
DATA, GT = '../input/Botswana.mat', '../input/Botswana_gt.mat'
CUBE_KEY, GT_KEY = None, None
p = use(BASELINES[0])
dc = p.cfg["data"]
cube, gt = p.hsi_data.load(DATA, GT, CUBE_KEY, GT_KEY)
data = p.hsi_data.build(cube, gt, split=dc["split"], seed=p.cfg["split_seed"], patch=dc["patch"],
                        pca=dc["pca"], frac=dc["frac"], block=dc["block"])
print("cube", cube.shape, "| classes", len(np.unique(gt[gt > 0])),
      "| train / val / test:", len(data["ytr"]), len(data["yva"]), len(data["yte"]))

## 3. The baselines

Training helpers first; then one section per baseline.

In [ ]:
# Training of one baseline: the package's own recipe (core.train_full: AdamW, linear warm-up then
# cosine, gradient clipping, best-validation checkpoint), on DEVICE (GPU on Colab).
EPOCHS = None   # None = the recipe of each package (config.yaml -> recipe.epochs); e.g. 30 for a quick run
RESULTS = {}

def show_architecture(name):
    """Paper-style block diagram (architecture.svg, drawn by DLNet from the spec)."""
    svg = OUT / name / "architecture.svg"
    if svg.exists():
        display(SVG(filename=str(svg)))

def show_layers(name, depth=2):
    """Layer-by-layer graph drawn by torchview (needs the graphviz program; Colab has it)."""
    p = use(name)
    model = p.model.build_model()
    kind, shape = p.cfg["arch"]["input"]["kind"], p.cfg["arch"]["input"]["dims"]
    try:
        from torchview import draw_graph
        if kind == "ids":
            x = torch.stack([torch.randint(0, c, (1,)) for c in shape], 1)
            g = draw_graph(model, input_data=x, depth=depth, expand_nested=True, graph_name=name)
        else:
            g = draw_graph(model, input_size=(1, *shape), depth=depth, expand_nested=True, graph_name=name)
        g.visual_graph.render(str(OUT / name / "layers"), format="svg", cleanup=True)
        display(SVG(filename=str(OUT / name / "layers.svg")))
    except Exception as e:  # torchview or graphviz missing, or a block it cannot trace
        print(f"torchview graph not drawn ({type(e).__name__}: {e}); the module tree instead:")
        print(model)

def train(name, epochs=None, seed=0):
    p = use(name)
    arch, recipe = p.cfg["arch"], p.cfg["recipe"]
    epochs = epochs or EPOCHS or recipe["epochs"]
    torch.manual_seed(seed)
    model = p.model.build_model(arch, data["in_dim"], data["out_dim"])
    n = sum(q.numel() for q in model.parameters())
    print(f"{name}: {n:,} parameters, {epochs} epochs, lr {arch['lr']}, wd {arch['wd']}, batch {arch['batch']}")
    t0 = time.time()
    tr, best_val = p.core.train_full(model, data, arch["lr"], arch["wd"], arch["batch"], epochs,
                                     seed=seed, device=DEVICE, recipe=recipe.get("name"))
    secs = time.time() - t0
    val, test = tr.evaluate("va"), tr.evaluate("te")
    p.core.save_artifacts(model.cpu(), data, p.dir)  # model.pt + preprocess.json next to model.py
    RESULTS[name] = {"params": n, "epochs": epochs, "seconds": round(secs, 1), "val": val, "test": test,
                     "history": getattr(tr, "history", [])}
    print(f"  {secs:.0f}s | val {val} | test {test}")
    plot_history(name)
    return RESULTS[name]

def plot_history(name):
    h = RESULTS.get(name, {}).get("history")
    if not h or plt is None:
        return
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].plot([r["epoch"] for r in h], [r["train_loss"] for r in h]); ax[0].set_title("training loss")
    ax[1].plot([r["epoch"] for r in h], [r["val"] for r in h], color="tab:orange")
    ax[1].set_title(f"validation {board['metric']}")
    for a in ax:
        a.set_xlabel("epoch"); a.grid(alpha=0.3)
    fig.suptitle(name); plt.tight_layout(); plt.show()

### top1_spec_fft2d

28,302 parameters; search: validation 0.9990 +- 0.0014 after 30 epochs (group 0: same group = indistinguishable within the noise). Block diagram (DLNet) and layer graph (torchview), then full training.

In [ ]:
NAME = 'top1_spec_fft2d'
show_architecture(NAME)
show_layers(NAME)

In [ ]:
train(NAME)

### top2_zoo_ffn_gated_attn_block_gfnet_block

119,950 parameters; search: validation 0.9990 +- 0.0014 after 30 epochs (group 0: same group = indistinguishable within the noise). Block diagram (DLNet) and layer graph (torchview), then full training.

In [ ]:
NAME = 'top2_zoo_ffn_gated_attn_block_gfnet_block'
show_architecture(NAME)
show_layers(NAME)

In [ ]:
train(NAME)

### top3_zoo_fca_nin_block

48,622 parameters; search: validation 0.9969 +- 0.0025 after 30 epochs (group 0: same group = indistinguishable within the noise). Block diagram (DLNet) and layer graph (torchview), then full training.

In [ ]:
NAME = 'top3_zoo_fca_nin_block'
show_architecture(NAME)
show_layers(NAME)

In [ ]:
train(NAME)

## 4. Comparison

In [ ]:
# 4. Comparison of the baselines (this notebook's training, same data and split)
metric = board["metric"]
rows = [(n, r["params"], r["epochs"], r["seconds"], r["val"].get("score"), r["test"].get("score"))
        for n, r in RESULTS.items()]
print(f"{'baseline':<45}{'params':>11}{'epochs':>8}{'sec':>8}{'val':>9}{'test':>9}")
for n, pa, ep, sec, v, t in sorted(rows, key=lambda r: -(r[4] or 0)):
    print(f"{n:<45}{pa:>11,}{ep:>8}{sec:>8}{v:>9.4f}{t:>9.4f}")
print("(score = the primary metric:", metric, "; regression scores are minus the RMSE)")
if plt is not None and RESULTS:
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    for n, r in RESULTS.items():
        if r["history"]:
            ax[0].plot([h["epoch"] for h in r["history"]], [h["val"] for h in r["history"]], label=n)
    ax[0].set_title(f"validation {metric} per epoch"); ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3)
    names = list(RESULTS)
    ax[1].barh(names, [RESULTS[n]["test"]["score"] for n in names], color="tab:green")
    ax[1].set_title(f"test {metric}"); ax[1].grid(alpha=0.3, axis="x")
    plt.tight_layout(); plt.show()

## 5. Reuse the saved baselines

In [ ]:
# 5. Load the saved baselines back, one by one, and use them (model.pt + preprocess.json)
def evaluate_saved(name):
    p = use(name)
    if not (p.dir / "model.pt").exists():
        print(f"{name}: no model.pt yet (train it in section 3)"); return None
    model, meta, cfg = p.inference.load(p.dir, "cpu")
    a = cfg["arch"]
    test = p.core.Trainer(model, data, lr=a["lr"], wd=a["wd"], batch=a["batch"]).evaluate("te")
    print(f"{name}: test {test}")
    return p, model.cpu().eval(), meta, cfg

for NAME in BASELINES:
    loaded = evaluate_saved(NAME)

In [ ]:
# Class map of the whole scene from the best saved baseline (predict.py does the same)
best = max(RESULTS, key=lambda n: RESULTS[n]["val"]["score"]) if RESULTS else BASELINES[0]
p, model, meta, cfg = evaluate_saved(best)
cmap = p.inference.predict_file(model, meta, cfg, DATA)
if plt is not None:
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(np.where(gt > 0, gt, np.nan), cmap="tab20"); ax[0].set_title("ground truth")
    ax[1].imshow(cmap, cmap="tab20"); ax[1].set_title(f"{best}: every pixel")
    ax[2].imshow(np.where(gt > 0, cmap, np.nan), cmap="tab20"); ax[2].set_title("labelled pixels")
    for a in ax: a.axis("off")
    plt.show()